# 08 — C-NMC classifier environment

**Purpose:** Check the existing Conda/Jupyter/CUDA environment and C-NMC path.

**Required inputs:** Existing leukemia-yolo environment; `cnmc_root` in config.local.yaml.

**Saved outputs:** outputs/classifier/environment_report.json.

## 1. Load central configuration

Set `cnmc_root` once in ignored `config.local.yaml`. Source images are never moved or changed.

In [7]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "classifier_config.py").is_file():
        sys.path.insert(0, str(candidate)); break
from src.classifier_config import classifier_config, classifier_output
cfg = classifier_config()
out = classifier_output(cfg)
print("Project:", cfg["root"])
print("Config:", cfg["config_path"])
print("C-NMC source:", cfg["cnmc_root"])

Project: D:\CellCancer\leukemia-yolo
Config: D:\CellCancer\leukemia-yolo\config.local.yaml
C-NMC source: D:\Datasets\C-NMC_Leukemia


## 2. Inspect hardware and packages

System RAM and dedicated NVIDIA VRAM are separate resources.

In [8]:
import importlib.metadata, platform, shutil, subprocess, sys
import pandas as pd
import psutil, torch, torchvision
from src.config import write_json
checks = []
def add(check, value, status): checks.append({"check": check, "value": str(value), "status": status})
add("Python executable", sys.executable, "PASS")
add("Python version", sys.version.split()[0], "PASS" if sys.version_info[:2] == (3, 11) else "WARNING")
add("Conda environment / Jupyter kernel", os.environ.get("CONDA_DEFAULT_ENV", "Not set"), "PASS" if os.environ.get("CONDA_DEFAULT_ENV") == "leukemia-yolo" else "WARNING")
add("OS / CPU", f"{platform.platform()} / {platform.processor()}", "PASS" if sys.platform == "win32" else "WARNING")
ram = psutil.virtual_memory()
add("System RAM total / available GiB", f"{ram.total/2**30:.2f} / {ram.available/2**30:.2f}", "PASS")
add("Output disk free GiB", f"{shutil.disk_usage(out).free/2**30:.2f}", "PASS")
add("PyTorch / Torchvision", f"{torch.__version__} / {torchvision.__version__}", "PASS")
add("CUDA build / available", f"{torch.version.cuda} / {torch.cuda.is_available()}", "PASS" if torch.cuda.is_available() else "ERROR")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    add("NVIDIA GPU", torch.cuda.get_device_name(0), "PASS")
    add("Dedicated VRAM total / free GiB", f"{total/2**30:.2f} / {free/2**30:.2f}", "PASS")
else: add("NVIDIA GPU", "CUDA unavailable; GPU training blocked", "ERROR")
for package in ("Pillow", "scikit-learn", "grad-cam", "PyYAML"):
    try: add(package, importlib.metadata.version(package), "PASS")
    except importlib.metadata.PackageNotFoundError: add(package, "missing", "ERROR")
from src.cnmc_data import discover_training_data
try:
    training_root, _ = discover_training_data(cfg["cnmc_root"])
    add("C-NMC verified training directory", training_root, "PASS")
except (FileNotFoundError, ValueError) as exc:
    add("C-NMC verified training directory", exc, "ERROR")
add("Existing YOLO source", cfg["root"] / "src" / "training.py", "PASS" if (cfg["root"] / "src" / "training.py").is_file() else "ERROR")
try:
    driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
    add("NVIDIA driver", driver.stdout.strip() or driver.stderr.strip(), "PASS" if driver.returncode == 0 else "WARNING")
except (OSError, subprocess.TimeoutExpired): add("NVIDIA driver", "nvidia-smi unavailable", "WARNING")
display(pd.DataFrame(checks))
write_json(out / "environment_report.json", {"checks": checks, "cuda_ready": torch.cuda.is_available(), "gpu_device": 0})

,check,value,status
0,Python executable,c:\Users\majid\.conda\envs\leukemia-yolo\pytho...,PASS
1,Python version,3.11.17,PASS
2,Conda environment / Jupyter kernel,Not set,WARNING
3,OS / CPU,Windows-10-10.0.26200-SP0 / Intel64 Family 6 M...,PASS
4,System RAM total / available GiB,15.73 / 2.42,PASS
5,Output disk free GiB,139.52,PASS
6,PyTorch / Torchvision,2.14.1+cu126 / 0.29.1+cu126,PASS
7,CUDA build / available,12.6 / True,PASS
8,NVIDIA GPU,NVIDIA GeForce RTX 3050 Ti Laptop GPU,PASS
9,Dedicated VRAM total / free GiB,4.00 / 3.21,PASS


## 3. Gate and saved report

A missing CUDA build is visible now and stops training later.

In [9]:
assert (out / "environment_report.json").is_file()
print("PASS — CUDA and report ready" if torch.cuda.is_available() else "ERROR — report saved; install compatible CUDA PyTorch before notebook 11")

PASS — CUDA and report ready


## Next

Open **09_CNMC_Dataset_Inspection.ipynb** after setting the real C-NMC path.